In [1]:
import time
from datetime import datetime
import pywifi
import json
from pathlib import Path
import tkinter as tk
from tkinter import ttk, messagebox, scrolledtext, simpledialog

class WiFiRSSICollector:
    def __init__(self):
        # Stockage : {room_name: {BSSID: {"ssid": str, "measurements": [{"rssi": int, "time": str}]}}}
        self.data_file = Path("wifi_measurements.json")
        self.data = self.load_data()
        
        # Salle actuelle
        self.current_room = None
        
        # Interface WiFi
        wifi = pywifi.PyWiFi()
        self.iface = wifi.interfaces()[0]
        
    def load_data(self):
        """Charge les données existantes"""
        if self.data_file.exists():
            try:
                with open(self.data_file, 'r', encoding='utf-8') as f:
                    return json.load(f)
            except json.JSONDecodeError:
                print("⚠️  Fichier JSON corrompu, création d'un nouveau fichier")
                return {}
        return {}
    
    def save_data(self):
        """Sauvegarde les données"""
        with open(self.data_file, 'w', encoding='utf-8') as f:
            json.dump(self.data, f, indent=2, ensure_ascii=False)
    
    def get_rooms(self):
        """Retourne la liste des salles"""
        return list(self.data.keys())
    
    def create_room(self, room_name):
        """Crée une nouvelle salle"""
        if not room_name or not room_name.strip():
            return False, "Le nom de la salle ne peut pas être vide"
        
        room_name = room_name.strip()
        
        if room_name in self.data:
            return False, "Cette salle existe déjà"
        
        self.data[room_name] = {}
        self.save_data()
        return True, f"Salle '{room_name}' créée"
    
    def delete_room(self, room_name):
        """Supprime une salle"""
        if room_name in self.data:
            del self.data[room_name]
            self.save_data()
            return True
        return False
    
    def set_current_room(self, room_name):
        """Définit la salle active"""
        if room_name in self.data:
            self.current_room = room_name
            return True
        return False
    
    def scan_networks(self, interval=2):
        """Scan les réseaux WiFi"""
        self.iface.scan()
        time.sleep(interval)
        return self.iface.scan_results()
    
    def add_measurement(self):
        """Ajoute une mesure pour la salle courante"""
        if not self.current_room:
            return 0, 0, "Aucune salle sélectionnée"
        
        networks = self.scan_networks()
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        measurements_added = 0
        
        for network in networks:
            bssid = network.bssid
            ssid = network.ssid if network.ssid else "Hidden"
            rssi = network.signal
            
            # Initialiser le vecteur si nouveau BSSID dans cette salle
            if bssid not in self.data[self.current_room]:
                self.data[self.current_room][bssid] = {
                    "ssid": ssid,
                    "measurements": []
                }
            
            # Ajouter la mesure
            self.data[self.current_room][bssid]["measurements"].append({
                "rssi": rssi,
                "time": timestamp
            })
            measurements_added += 1
        
        self.save_data()
        return measurements_added, len(networks), "OK"
    
    def get_room_statistics(self, room_name):
        """Statistiques pour une salle"""
        if room_name not in self.data:
            return 0, 0
        
        room_data = self.data[room_name]
        
        if not isinstance(room_data, dict):
            return 0, 0
        
        total_bssids = len(room_data)
        total_measurements = 0
        
        for bssid, info in room_data.items():
            if isinstance(info, dict) and "measurements" in info:
                total_measurements += len(info["measurements"])
        
        return total_bssids, total_measurements
    
    def get_global_statistics(self):
        """Statistiques globales"""
        total_rooms = len(self.data)
        total_bssids = set()
        total_measurements = 0
        
        for room_name, room_data in self.data.items():
            if not isinstance(room_data, dict):
                continue
                
            for bssid, info in room_data.items():
                total_bssids.add(bssid)
                if isinstance(info, dict) and "measurements" in info:
                    total_measurements += len(info["measurements"])
        
        return total_rooms, len(total_bssids), total_measurements
    
    def get_room_details(self, room_name):
        """Détails des mesures par AP pour une salle"""
        if room_name not in self.data:
            return []
        
        details = []
        room_data = self.data[room_name]
        
        if not isinstance(room_data, dict):
            return []
        
        for bssid, info in room_data.items():
            if isinstance(info, dict) and "measurements" in info:
                rssi_values = [m["rssi"] for m in info["measurements"]]
                avg_rssi = sum(rssi_values) / len(rssi_values) if rssi_values else 0
                
                details.append({
                    "bssid": bssid,
                    "ssid": info.get("ssid", "Unknown"),
                    "count": len(info["measurements"]),
                    "avg_rssi": avg_rssi,
                    "min_rssi": min(rssi_values) if rssi_values else 0,
                    "max_rssi": max(rssi_values) if rssi_values else 0
                })
        
        return sorted(details, key=lambda x: x["count"], reverse=True)
    
    def export_to_csv(self, filename="wifi_data.csv"):
        """Export complet avec salles"""
        with open(filename, 'w', encoding='utf-8') as f:
            f.write("Room,BSSID,SSID,RSSI(dBm),Time\n")
            for room_name, room_data in self.data.items():
                if not isinstance(room_data, dict):
                    continue
                for bssid, info in room_data.items():
                    if not isinstance(info, dict) or "measurements" not in info:
                        continue
                    for m in info["measurements"]:
                        f.write(f"{room_name},{bssid},{info['ssid']},{m['rssi']},{m['time']}\n")
    
    def export_room_to_csv(self, room_name, filename=None):
        """Export d'une seule salle"""
        if room_name not in self.data:
            return False
        
        if not filename:
            filename = f"wifi_{room_name.replace(' ', '_')}.csv"
        
        with open(filename, 'w', encoding='utf-8') as f:
            f.write("BSSID,SSID,RSSI(dBm),Time\n")
            room_data = self.data[room_name]
            
            if not isinstance(room_data, dict):
                return False
            
            for bssid, info in room_data.items():
                if not isinstance(info, dict) or "measurements" not in info:
                    continue
                for m in info["measurements"]:
                    f.write(f"{bssid},{info['ssid']},{m['rssi']},{m['time']}\n")
        return True


class WiFiCollectorGUI:
    def __init__(self):
        self.collector = WiFiRSSICollector()
        self.setup_ui()
        
    def setup_ui(self):
        """Création de l'interface"""
        self.root = tk.Tk()
        self.root.title("WiFi RSSI Collector - Par Salle")
        self.root.geometry("900x700")
        
        # Frame principal
        main_frame = ttk.Frame(self.root, padding="10")
        main_frame.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # === Section gestion des salles ===
        room_frame = ttk.LabelFrame(main_frame, text="🏢 Gestion des salles", padding="15")
        room_frame.grid(row=0, column=0, columnspan=2, sticky=(tk.W, tk.E), pady=5)
        
        # Sélection de salle
        ttk.Label(room_frame, text="Salle active:", font=('Arial', 10, 'bold')).grid(row=0, column=0, sticky=tk.W, padx=5)
        
        self.room_var = tk.StringVar()
        self.room_combo = ttk.Combobox(room_frame, textvariable=self.room_var, width=30, state='readonly', font=('Arial', 10))
        self.room_combo.grid(row=0, column=1, padx=5)
        self.room_combo.bind('<<ComboboxSelected>>', self.on_room_selected)
        
        # Boutons gestion salles
        ttk.Button(room_frame, text="➕ Nouvelle salle", command=self.create_room, width=15).grid(row=0, column=2, padx=5)
        ttk.Button(room_frame, text="🗑️ Supprimer", command=self.delete_room, width=15).grid(row=0, column=3, padx=5)
        
        # Indicateur salle courante
        self.room_status_label = ttk.Label(
            room_frame, 
            text="❌ Aucune salle sélectionnée", 
            foreground="red", 
            font=('Arial', 11, 'bold')
        )
        self.room_status_label.grid(row=1, column=0, columnspan=4, pady=10)
        
        # === Section MESURE ===
        measure_frame = ttk.LabelFrame(main_frame, text="📡 Mesure WiFi", padding="20")
        measure_frame.grid(row=1, column=0, columnspan=2, sticky=(tk.W, tk.E), pady=10)
        
        # Grand bouton de mesure
        self.measure_btn = tk.Button(
            measure_frame, 
            text="📡 CAPTURER LES RSSI\n(Appuyez sur Espace)", 
            command=self.capture_measurement,
            state='disabled',
            font=('Arial', 14, 'bold'),
            bg='#4CAF50',
            fg='white',
            height=3,
            width=30,
            cursor='hand2'
        )
        self.measure_btn.pack(pady=10)
        
        # Info mesure en cours
        self.measure_info_label = ttk.Label(
            measure_frame, 
            text="Sélectionnez une salle pour commencer", 
            font=('Arial', 9, 'italic'),
            foreground='gray'
        )
        self.measure_info_label.pack()
        
        # Bind Espace
        self.root.bind('<space>', lambda e: self.capture_measurement())
        
        # === Section détails de la salle ===
        details_frame = ttk.LabelFrame(main_frame, text="📊 Détails de la salle courante", padding="10")
        details_frame.grid(row=2, column=0, columnspan=2, sticky=(tk.W, tk.E, tk.N, tk.S), pady=5)
        
        # Tableau des AP
        self.details_text = scrolledtext.ScrolledText(details_frame, height=10, width=100, font=('Courier', 9))
        self.details_text.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # === Section statistiques ===
        stats_frame = ttk.LabelFrame(main_frame, text="📈 Statistiques globales", padding="10")
        stats_frame.grid(row=3, column=0, columnspan=2, sticky=(tk.W, tk.E), pady=5)
        
        self.stats_label = ttk.Label(stats_frame, text="", font=('Arial', 10))
        self.stats_label.grid(row=0, column=0, sticky=tk.W)
        
        # === Section log ===
        log_frame = ttk.LabelFrame(main_frame, text="📝 Historique", padding="10")
        log_frame.grid(row=4, column=0, columnspan=2, sticky=(tk.W, tk.E, tk.N, tk.S), pady=5)
        
        self.log_text = scrolledtext.ScrolledText(log_frame, height=8, width=100, font=('Courier', 9))
        self.log_text.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # === Boutons actions ===
        btn_frame = ttk.Frame(main_frame)
        btn_frame.grid(row=5, column=0, columnspan=2, pady=10)
        
        ttk.Button(btn_frame, text="💾 Export salle", command=self.export_room, width=15).pack(side=tk.LEFT, padx=5)
        ttk.Button(btn_frame, text="💾 Export tout", command=self.export_all, width=15).pack(side=tk.LEFT, padx=5)
        ttk.Button(btn_frame, text="🔄 Rafraîchir", command=self.update_all, width=15).pack(side=tk.LEFT, padx=5)
        ttk.Button(btn_frame, text="❌ Quitter", command=self.root.quit, width=15).pack(side=tk.LEFT, padx=5)
        
        # Configuration du redimensionnement
        self.root.columnconfigure(0, weight=1)
        self.root.rowconfigure(0, weight=1)
        main_frame.columnconfigure(0, weight=1)
        main_frame.rowconfigure(2, weight=1)
        main_frame.rowconfigure(4, weight=1)
        
        # Initialiser
        self.refresh_rooms()
        self.update_all()
        
    def refresh_rooms(self):
        """Met à jour la liste des salles"""
        rooms = self.collector.get_rooms()
        self.room_combo['values'] = rooms
        
        if rooms and not self.collector.current_room:
            self.room_combo.current(0)
            self.on_room_selected(None)
        elif not rooms:
            # Créer une salle par défaut si aucune n'existe
            self.collector.create_room("Salle_01")
            self.refresh_rooms()
    
    def on_room_selected(self, event):
        """Quand une salle est sélectionnée"""
        room_name = self.room_var.get()
        if room_name:
            self.collector.set_current_room(room_name)
            self.room_status_label.config(
                text=f"✅ Salle active: {room_name}",
                foreground="green"
            )
            self.measure_btn.config(state='normal', bg='#4CAF50')
            self.measure_info_label.config(text="Prêt à mesurer - Appuyez sur ESPACE ou cliquez", foreground='green')
            self.update_all()
            self.log_text.insert('1.0', f"🏢 Salle '{room_name}' sélectionnée\n")
    
    def create_room(self):
        """Crée une nouvelle salle"""
        room_name = simpledialog.askstring(
            "Nouvelle salle",
            "Nom de la salle (ex: Salle_A101, Bureau_12, Couloir_Est):",
            parent=self.root
        )
        
        if room_name:
            success, message = self.collector.create_room(room_name)
            
            if success:
                self.refresh_rooms()
                self.room_var.set(room_name)
                self.on_room_selected(None)
                self.log_text.insert('1.0', f"✅ {message}\n")
            else:
                messagebox.showwarning("Attention", message)
    
    def delete_room(self):
        """Supprime la salle courante"""
        room_name = self.room_var.get()
        if not room_name:
            messagebox.showwarning("Attention", "Aucune salle sélectionnée")
            return
        
        confirm = messagebox.askyesno(
            "Confirmation",
            f"Supprimer la salle '{room_name}' et toutes ses mesures ?"
        )
        
        if confirm:
            self.collector.delete_room(room_name)
            self.collector.current_room = None
            self.refresh_rooms()
            self.room_status_label.config(
                text="❌ Aucune salle sélectionnée",
                foreground="red"
            )
            self.measure_btn.config(state='disabled', bg='gray')
            self.measure_info_label.config(text="Sélectionnez une salle pour commencer", foreground='gray')
            self.log_text.insert('1.0', f"🗑️ Salle '{room_name}' supprimée\n")
            self.update_all()
    
    def capture_measurement(self):
        """Capture une mesure"""
        if not self.collector.current_room:
            messagebox.showwarning("Attention", "Sélectionnez d'abord une salle")
            return
        
        # Désactiver pendant la mesure
        self.measure_btn.config(state='disabled', text="🔄 SCAN EN COURS...", bg='orange')
        self.measure_info_label.config(text="Scan WiFi en cours (2 secondes)...", foreground='orange')
        self.root.update()
        
        try:
            num_measurements, num_networks, status = self.collector.add_measurement()
            
            if status == "OK":
                msg = f"✅ [{self.collector.current_room}] {num_networks} réseaux détectés et enregistrés\n"
                self.log_text.insert('1.0', msg)
                
                self.update_all()
                
                # Feedback visuel
                self.measure_btn.config(bg='#2E7D32')  # Vert foncé
                self.root.after(500, lambda: self.measure_btn.config(bg='#4CAF50'))  # Retour vert normal
            else:
                messagebox.showerror("Erreur", status)
                
        except Exception as e:
            messagebox.showerror("Erreur", f"Erreur: {str(e)}")
        finally:
            self.measure_btn.config(
                state='normal', 
                text="📡 CAPTURER LES RSSI\n(Appuyez sur Espace)"
            )
            self.measure_info_label.config(text="Prêt à mesurer - Appuyez sur ESPACE ou cliquez", foreground='green')
    
    def update_room_details(self):
        """Met à jour les détails de la salle courante"""
        self.details_text.delete('1.0', tk.END)
        
        if not self.collector.current_room:
            self.details_text.insert('1.0', "Aucune salle sélectionnée")
            return
        
        details = self.collector.get_room_details(self.collector.current_room)
        
        if not details:
            self.details_text.insert('1.0', "Aucune mesure enregistrée pour cette salle")
            return
        
        # En-tête
        header = f"{'SSID':<25} {'BSSID':<20} {'Mesures':<10} {'RSSI Moy':<12} {'Min':<8} {'Max':<8}\n"
        header += "=" * 90 + "\n"
        self.details_text.insert('1.0', header)
        
        # Données
        for detail in details:
            line = (
                f"{detail['ssid']:<25} "
                f"{detail['bssid']:<20} "
                f"{detail['count']:<10} "
                f"{detail['avg_rssi']:>6.1f} dBm   "
                f"{detail['min_rssi']:>4} dBm  "
                f"{detail['max_rssi']:>4} dBm\n"
            )
            self.details_text.insert(tk.END, line)
    
    def update_stats(self):
        """Met à jour les statistiques"""
        total_rooms, total_bssids, total_measurements = self.collector.get_global_statistics()
        
        stats_text = f"🏢 {total_rooms} salles | 📡 {total_bssids} points d'accès uniques | 📊 {total_measurements} mesures totales"
        
        if self.collector.current_room:
            room_bssids, room_measurements = self.collector.get_room_statistics(self.collector.current_room)
            stats_text += f"\n📍 Salle courante: {room_bssids} AP, {room_measurements} mesures"
        
        self.stats_label.config(text=stats_text)
    
    def update_all(self):
        """Met à jour toute l'interface"""
        self.update_stats()
        self.update_room_details()
    
    def export_room(self):
        """Export de la salle courante"""
        if not self.collector.current_room:
            messagebox.showwarning("Attention", "Aucune salle sélectionnée")
            return
        
        try:
            filename = f"wifi_{self.collector.current_room.replace(' ', '_')}.csv"
            self.collector.export_room_to_csv(self.collector.current_room, filename)
            messagebox.showinfo("Export réussi", f"Salle exportée dans {filename}")
            self.log_text.insert('1.0', f"💾 Export salle '{self.collector.current_room}' → {filename}\n")
        except Exception as e:
            messagebox.showerror("Erreur", f"Erreur: {str(e)}")
    
    def export_all(self):
        """Export de toutes les salles"""
        try:
            self.collector.export_to_csv()
            messagebox.showinfo("Export réussi", "Toutes les données exportées dans wifi_data.csv")
            self.log_text.insert('1.0', "💾 Export complet → wifi_data.csv\n")
        except Exception as e:
            messagebox.showerror("Erreur", f"Erreur: {str(e)}")
    
    def run(self):
        """Lance l'application"""
        self.root.mainloop()


if __name__ == "__main__":
    app = WiFiCollectorGUI()
    app.run()
